# Neural Networks

In [40]:
# %pip install pandas matplotlib seaborn scikit-learn tensorflow

In [41]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

data = pd.read_csv(r'../data/cleaned_data.csv')

train = data[data['order_year']<=2021]
test = data[data['order_year']>2021]

train_cats = set(train['Category'].unique())
test_cats = set(test['Category'].unique())
unseen = test_cats - train_cats
if len(unseen) > 0:
    test = test[~test['Category'].isin(unseen)]

data_limited_train = train.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)
data_limited_test = test.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)

X_train = data_limited_train.drop('Category', axis=1)
y_train = data_limited_train['Category']
X_train_scaled = StandardScaler().fit_transform(X_train)

X_test = data_limited_test.drop('Category', axis=1)
y_test = data_limited_test['Category']
X_test_scaled = StandardScaler().fit_transform(X_test)

In [42]:
X_train_scaled.shape[1]

153

In [43]:
y_train.max()

np.int64(1624)

In [44]:
import tensorflow as tf

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(153, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

c:\Users\shaha\Documents\1. SU Masters\Code Space\group-project-bas-team\.venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [45]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 300)            │        46,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 240,425 (939.16 KB)

 Trainable params: 240,425 (939.16 KB)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [47]:
history = model.fit(X_train_scaled, y_train, epochs=30)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.0586 - loss: 6.4280
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.0691 - loss: 6.0548
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.0769 - loss: 5.9404
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.0833 - loss: 5.8429
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.0879 - loss: 5.7534
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.0912 - loss: 5.6704
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.0943 - loss: 5.5955
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.0974 - loss: 5.5294
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.0999 - loss: 5.4713
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.1023 - loss: 5.4201
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.1044 - loss: 5.3745
Epoch 12/30
3552/3552 ━━━━━━━

In [48]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1355/1355 - 2s - 1ms/step - accuracy: 0.0667 - loss: 5.9930

Test accuracy: 0.06669897586107254


Our initial neural network has an accuracy of about 7.0%, which is an improvement from our random forest model.

Next, we will attempt to build a wide & deep neural network

In [49]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

normalization_layer = tf.keras.layers.Normalization()

# two Dense layers with 30 neurons each, using the ReLU activation function
hidden_layer1 = tf.keras.layers.Dense(30, activation="relu")
hidden_layer2 = tf.keras.layers.Dense(30, activation="relu")

concat_layer = tf.keras.layers.Concatenate()

output_layer = tf.keras.layers.Dense(1625)


input_ = tf.keras.layers.Input(shape=X_train_scaled.shape[1:])

normalized = normalization_layer(input_)

hidden1 = hidden_layer1(normalized)
hidden2 = hidden_layer2(hidden1)

concat = concat_layer([normalized, hidden2])

output = output_layer(concat)

model = tf.keras.Model(inputs=[input_], outputs=[output])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 153)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 153)       │        307 │ input_layer[0][0] │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30)        │      4,620 │ normalization[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 30)        │        930 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 183)       │          0 │ normalization[0]… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1625)      │    299,000 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 304,857 (1.16 MB)

 Trainable params: 304,550 (1.16 MB)

 Non-trainable params: 307 (1.20 KB)

In [50]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

normalization_layer.adapt(X_train_scaled)
history = model.fit(X_train_scaled, y_train, epochs=20)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.0155 - loss: 10.1714
Epoch 2/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.0014 - loss: 9.3142
Epoch 3/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.0012 - loss: 9.0045
Epoch 4/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.0018 - loss: 8.9071
Epoch 5/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.0026 - loss: 8.8101
Epoch 6/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.0015 - loss: 8.7492
Epoch 7/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 7.7427e-04 - loss: 8.8597
Epoch 8/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 4.1353e-04 - loss: 8.9321
Epoch 9/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 4.2233e-04 - loss: 8.8940
Epoch 10/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 4.1353e-04 - loss: 8.8825
Epoch 11/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 4.3993e-04 - loss: 8.8596
Epoch 

Very poor accuracy and high loss - need to revise

In [51]:
X_train.columns

Index(['Purchase Price Per Unit', 'Quantity', 'age', 'hispanic', 'education',
       'income', 'howmany', 'hh-size', 'how-oft', 'order_month',
       ...
       'life-changes_Lost a job ,Divorce',
       'life-changes_Lost a job ,Divorce,Moved place of residence',
       'life-changes_Lost a job ,Had a child',
       'life-changes_Lost a job ,Moved place of residence',
       'life-changes_Lost a job ,Moved place of residence,Became pregnant',
       'life-changes_Lost a job ,Moved place of residence,Became pregnant,Had a child',
       'life-changes_Lost a job ,Moved place of residence,Had a child',
       'life-changes_Moved place of residence',
       'life-changes_Moved place of residence,Became pregnant,Had a child',
       'life-changes_Moved place of residence,Had a child'],
      dtype='str', length=153)

In [52]:
corr_matrix = X_train.corr().abs()
top_corr = corr_matrix.unstack().sort_values(ascending=False)
top_corr = top_corr[top_corr < 1].drop_duplicates()
print(top_corr.head(20))

diabetes_No                                 diabetes_Yes                 0.999812
wheelchair_Yes                              wheelchair_No                0.998180
state_Alabama                               Shipping Address State_AL    0.989636
Shipping Address State_ME                   state_Maine                  0.987357
state_Iowa                                  Shipping Address State_IA    0.982405
sexual-orientation_heterosexual (straight)  sexual-orientation_LGBTQ+    0.978118
Shipping Address State_NE                   state_Nebraska               0.973867
Shipping Address State_MI                   state_Michigan               0.972118
state_Idaho                                 Shipping Address State_ID    0.971907
marijuana_Yes                               marijuana_No                 0.969490
Shipping Address State_MO                   state_Missouri               0.969445
Shipping Address State_KY                   state_Kentucky               0.966610
state_North Caro

In [53]:
# It appears that states and shipping address states are highly correlated - we will remove all shipping address state variables
# Same with the diabetes_No, wheelchair_No, Marijuana_No, sexual-orientation_heterosexual (straight), gender_female, and alcohol_No variables, which are all very highly correlated with other variables
# this will significantly limit the number of variables to subset, making it easier to build our wide and deep neural network

X_train = X_train.drop(X_train.filter(regex='^Shipping Address').columns, axis=1)
X_test = X_test.drop(X_test.filter(regex='^Shipping Address').columns, axis=1)

In [54]:
X_train = X_train.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])
X_test = X_test.drop(columns = ['diabetes_No', 'wheelchair_No', 'marijuana_No', 'sexual-orientation_heterosexual (straight)', 
                                  'gender_Female', 'alcohol_No'])

In [55]:
X_train.columns

Index(['Purchase Price Per Unit', 'Quantity', 'age', 'hispanic', 'education',
       'income', 'howmany', 'hh-size', 'how-oft', 'order_month', 'order_day',
       'order_year', 'cigarettes_I stopped in the recent past',
       'cigarettes_No', 'cigarettes_Yes',
       'alcohol_I stopped in the recent past', 'alcohol_Yes',
       'marijuana_I stopped in the recent past', 'marijuana_Yes',
       'diabetes_Yes', 'wheelchair_Yes', 'gender_Male', 'gender_Other',
       'sexual-orientation_LGBTQ+', 'state_Alabama', 'state_Arizona',
       'state_Arkansas', 'state_California', 'state_Colorado',
       'state_Connecticut', 'state_Delaware', 'state_Florida', 'state_Georgia',
       'state_Hawaii', 'state_Idaho', 'state_Illinois', 'state_Indiana',
       'state_Iowa', 'state_Kansas', 'state_Kentucky', 'state_Louisiana',
       'state_Maine', 'state_Maryland', 'state_Massachusetts',
       'state_Michigan', 'state_Minnesota', 'state_Missouri', 'state_Nebraska',
       'state_Nevada', 'state_New H

In [56]:
X_test.columns

Index(['Purchase Price Per Unit', 'Quantity', 'age', 'hispanic', 'education',
       'income', 'howmany', 'hh-size', 'how-oft', 'order_month', 'order_day',
       'order_year', 'cigarettes_I stopped in the recent past',
       'cigarettes_No', 'cigarettes_Yes',
       'alcohol_I stopped in the recent past', 'alcohol_Yes',
       'marijuana_I stopped in the recent past', 'marijuana_Yes',
       'diabetes_Yes', 'wheelchair_Yes', 'gender_Male', 'gender_Other',
       'sexual-orientation_LGBTQ+', 'state_Alabama', 'state_Arizona',
       'state_Arkansas', 'state_California', 'state_Colorado',
       'state_Connecticut', 'state_Delaware', 'state_Florida', 'state_Georgia',
       'state_Hawaii', 'state_Idaho', 'state_Illinois', 'state_Indiana',
       'state_Iowa', 'state_Kansas', 'state_Kentucky', 'state_Louisiana',
       'state_Maine', 'state_Maryland', 'state_Massachusetts',
       'state_Michigan', 'state_Minnesota', 'state_Missouri', 'state_Nebraska',
       'state_Nevada', 'state_New H

In [57]:
X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit_transform(X_test)

In [58]:
# Rerunning our neural networks to see if removing highly correlated variables improved our accuracy

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(98, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

c:\Users\shaha\Documents\1. SU Masters\Code Space\group-project-bas-team\.venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [59]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 300)            │        29,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 223,925 (874.71 KB)

 Trainable params: 223,925 (874.71 KB)

 Non-trainable params: 0 (0.00 B)

In [60]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [61]:
history = model.fit(X_train_scaled, y_train, epochs=30)

Epoch 1/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.0568 - loss: 6.4850
Epoch 2/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.0681 - loss: 6.0766
Epoch 3/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.0747 - loss: 5.9688
Epoch 4/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.0808 - loss: 5.8718
Epoch 5/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.0857 - loss: 5.7830
Epoch 6/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.0901 - loss: 5.7030
Epoch 7/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.0928 - loss: 5.6317
Epoch 8/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.0957 - loss: 5.5687
Epoch 9/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.0984 - loss: 5.5130
Epoch 10/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.1003 - loss: 5.4638
Epoch 11/30
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.1024 - loss: 5.4199
Epoch 12/30
3552/3552 ━━━━━━━━

In [62]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1355/1355 - 3s - 2ms/step - accuracy: 0.0678 - loss: 5.9600

Test accuracy: 0.06776062399148941


In [63]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

normalization_layer = tf.keras.layers.Normalization()

# two Dense layers with 30 neurons each, using the ReLU activation function
hidden_layer1 = tf.keras.layers.Dense(30, activation="relu")
hidden_layer2 = tf.keras.layers.Dense(30, activation="relu")

concat_layer = tf.keras.layers.Concatenate()

output_layer = tf.keras.layers.Dense(1625)


input_ = tf.keras.layers.Input(shape=X_train_scaled.shape[1:])

normalized = normalization_layer(input_)

hidden1 = hidden_layer1(normalized)
hidden2 = hidden_layer2(hidden1)

concat = concat_layer([normalized, hidden2])

output = output_layer(concat)

# Finally create the model, specifying inputs and outputs
model = tf.keras.Model(inputs=[input_], outputs=[output])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 98)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 98)        │        197 │ input_layer[0][0] │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30)        │      2,970 │ normalization[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 30)        │        930 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ normalization[0]… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1625)      │    209,625 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 213,722 (834.86 KB)

 Trainable params: 213,525 (834.08 KB)

 Non-trainable params: 197 (792.00 B)

In [64]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

normalization_layer.adapt(X_train_scaled)
history = model.fit(X_train_scaled, y_train, epochs=20)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

Epoch 1/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.0010 - loss: 8.8192
Epoch 2/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 5.1911e-04 - loss: 8.0335
Epoch 3/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 5.3671e-04 - loss: 7.9418
Epoch 4/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 4.3993e-04 - loss: 7.8710
Epoch 5/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 7.9187e-05 - loss: 7.9661
Epoch 6/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 8.7986e-05 - loss: 7.9119
Epoch 7/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 7.9187e-05 - loss: 7.8931
Epoch 8/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 1.1438e-04 - loss: 7.8468
Epoch 9/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 1.2318e-04 - loss: 7.7888
Epoch 10/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 1.2318e-04 - loss: 7.7674
Epoch 11/20
3552/3552 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 2.1117e-04 - lo

Next, we will try subsetting features so that the wide and deep components are trained on different features

In [65]:
# Wide: transactional features with likely direct/linear effect on category
wide_cols = [
    'Purchase Price Per Unit', 'Quantity',
    'order_month', 'order_day', 'order_year'
]

# Deep: demographic + behavioral features where interactions matter
deep_cols = [c for c in X_train.columns if c not in wide_cols]

print(f"Wide features : {len(wide_cols)}")
print(f"Deep features : {len(deep_cols)}")

Wide features : 5
Deep features : 93


In [66]:
wide_scaler = StandardScaler()
deep_scaler  = StandardScaler()

X_train_wide = wide_scaler.fit_transform(X_train[wide_cols])
X_test_wide  = wide_scaler.transform(X_test[wide_cols])

X_train_deep = deep_scaler.fit_transform(X_train[deep_cols])
X_test_deep  = deep_scaler.transform(X_test[deep_cols])

In [67]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(64, activation="relu", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="relu", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(64, activation="relu", name="deep3")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 64)        │        384 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ deep3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 128)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1625)      │    209,625 │ wide_deep_concat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 276,761 (1.06 MB)

 Trainable params: 275,993 (1.05 MB)

 Non-trainable params: 768 (3.00 KB)

In [68]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [69]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=100, callbacks=callbacks, validation_split=0.3)

Epoch 1/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0667 - loss: 6.0640 - val_accuracy: 0.0653 - val_loss: 6.0505 - learning_rate: 0.0010
Epoch 2/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0818 - loss: 5.6754 - val_accuracy: 0.0658 - val_loss: 6.0678 - learning_rate: 0.0010
Epoch 3/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0883 - loss: 5.4982 - val_accuracy: 0.0688 - val_loss: 6.1119 - learning_rate: 0.0010
Epoch 4/100
2480/2487 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0910 - loss: 5.3715
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.0928 - loss: 5.3715 - val_accuracy: 0.0643 - val_loss: 6.1790 - learning_rate: 0.0010
Epoch 5/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0960 - loss: 5.2525 - val_accuracy: 0.0646 - val_loss: 6.2273 - learning_rate: 5.0000e-04
Epoch 6/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accura

In [70]:
# Test model accuracy
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 2s - 2ms/step - accuracy: 0.0538 - loss: 6.0726

Test accuracy: 0.0538


In [71]:
# Same model as above, but applying early stopping

tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(64, activation="relu", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="relu", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(64, activation="relu", name="deep3")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 64)        │        384 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ deep3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 128)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1625)      │    209,625 │ wide_deep_concat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 276,761 (1.06 MB)

 Trainable params: 275,993 (1.05 MB)

 Non-trainable params: 768 (3.00 KB)

In [72]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

In [73]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=400, callbacks=early_stop, validation_split=0.3)

Epoch 1/400
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0660 - loss: 6.0717 - val_accuracy: 0.0626 - val_loss: 6.0642
Epoch 2/400
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0805 - loss: 5.6834 - val_accuracy: 0.0613 - val_loss: 6.0827
Epoch 3/400
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0876 - loss: 5.5061 - val_accuracy: 0.0622 - val_loss: 6.1338
Epoch 4/400
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0923 - loss: 5.3791 - val_accuracy: 0.0602 - val_loss: 6.2164
Epoch 5/400
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0945 - loss: 5.2885 - val_accuracy: 0.0607 - val_loss: 6.2814
Epoch 6/400
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0963 - loss: 5.2203 - val_accuracy: 0.0592 - val_loss: 6.3231


In [74]:
# Test model accuracy
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 2s - 2ms/step - accuracy: 0.0549 - loss: 6.0745

Test accuracy: 0.0549


In [75]:
# Trying more layers and more neurons per layer

tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(256, activation="relu", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="relu", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep3")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep4")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep5")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep6")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 128)       │     16,512 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep3[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep4 (Dense)       │ (None, 128)       │     16,512 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep4[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep5 (Dense)       │ (None, 128)       │     16,512 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep5[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep6 (Dense)       │ (None, 128)       │     16,512 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 256)       │      1,536 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ deep6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 384)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_5[0][0] 

 Total params: 753,241 (2.87 MB)

 Trainable params: 751,705 (2.87 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [76]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

early_stop=tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

In [77]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=100, callbacks=early_stop, validation_split=0.3)


Epoch 1/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0617 - loss: 6.1041 - val_accuracy: 0.0629 - val_loss: 5.9946
Epoch 2/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.0723 - loss: 5.7868 - val_accuracy: 0.0660 - val_loss: 6.0053
Epoch 3/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0780 - loss: 5.6463 - val_accuracy: 0.0642 - val_loss: 6.0682
Epoch 4/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.0809 - loss: 5.5399 - val_accuracy: 0.0628 - val_loss: 6.1590
Epoch 5/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0845 - loss: 5.4512 - val_accuracy: 0.0583 - val_loss: 6.2540
Epoch 6/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.0872 - loss: 5.3776 - val_accuracy: 0.0588 - val_loss: 6.3343
Epoch 7/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.0890 - loss: 5.3177 - val_accuracy: 0.0567 - val_loss: 6.4690


In [78]:
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 3s - 2ms/step - accuracy: 0.0543 - loss: 6.1420

Test accuracy: 0.0543


In [79]:
# Same model, but increasing number of epochs and using early stopping

tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(256, activation="relu", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="relu", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep3")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep4")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep5")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="relu", name="deep6")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 128)       │     16,512 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep3[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep4 (Dense)       │ (None, 128)       │     16,512 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep4[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep5 (Dense)       │ (None, 128)       │     16,512 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep5[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep6 (Dense)       │ (None, 128)       │     16,512 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 256)       │      1,536 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ deep6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 384)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_5[0][0] 

 Total params: 753,241 (2.87 MB)

 Trainable params: 751,705 (2.87 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [80]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

In [81]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=400, callbacks=early_stop, validation_split=0.2)

Epoch 1/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - accuracy: 0.0606 - loss: 6.0995 - val_accuracy: 0.0650 - val_loss: 5.9303
Epoch 2/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0708 - loss: 5.8006 - val_accuracy: 0.0683 - val_loss: 5.9096
Epoch 3/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0755 - loss: 5.6656 - val_accuracy: 0.0688 - val_loss: 5.9760
Epoch 4/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0794 - loss: 5.5648 - val_accuracy: 0.0667 - val_loss: 6.0541
Epoch 5/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0826 - loss: 5.4837 - val_accuracy: 0.0692 - val_loss: 6.1213
Epoch 6/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0841 - loss: 5.4195 - val_accuracy: 0.0650 - val_loss: 6.2114
Epoch 7/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0868 - loss: 5.3687 - val_accuracy: 0.0673 - val_loss: 6.2640
Epoch 8/400
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0873 -

In [82]:
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 2s - 2ms/step - accuracy: 0.0584 - loss: 6.7342

Test accuracy: 0.0584


Attempting different activation functions

In [83]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(64, activation="elu", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="elu", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="elu", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(64, activation="elu", name="deep3")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 64)        │        384 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ deep3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 128)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1625)      │    209,625 │ wide_deep_concat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 276,761 (1.06 MB)

 Trainable params: 275,993 (1.05 MB)

 Non-trainable params: 768 (3.00 KB)

In [84]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [85]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=100, callbacks = callbacks, validation_split=0.3)

Epoch 1/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.0705 - loss: 6.0278 - val_accuracy: 0.0567 - val_loss: 6.0996 - learning_rate: 0.0010
Epoch 2/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.0838 - loss: 5.6334 - val_accuracy: 0.0566 - val_loss: 6.1052 - learning_rate: 0.0010
Epoch 3/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0878 - loss: 5.5087 - val_accuracy: 0.0562 - val_loss: 6.1247 - learning_rate: 0.0010
Epoch 4/100
2484/2487 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0884 - loss: 5.4475
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0896 - loss: 5.4370 - val_accuracy: 0.0542 - val_loss: 6.1457 - learning_rate: 0.0010
Epoch 5/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0938 - loss: 5.3213 - val_accuracy: 0.0569 - val_loss: 6.1709 - learning_rate: 5.0000e-04
Epoch 6/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accurac

In [86]:
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 2s - 2ms/step - accuracy: 0.0515 - loss: 6.0751

Test accuracy: 0.0515


In [87]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(64, activation="selu", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="selu", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="selu", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(64, activation="selu", name="deep3")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 64)        │        384 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ deep3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 128)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1625)      │    209,625 │ wide_deep_concat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 276,761 (1.06 MB)

 Trainable params: 275,993 (1.05 MB)

 Non-trainable params: 768 (3.00 KB)

In [88]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [89]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=100, callbacks = callbacks, validation_split=0.3)

Epoch 1/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0702 - loss: 6.0227 - val_accuracy: 0.0588 - val_loss: 6.0977 - learning_rate: 0.0010
Epoch 2/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0828 - loss: 5.6441 - val_accuracy: 0.0597 - val_loss: 6.0983 - learning_rate: 0.0010
Epoch 3/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0866 - loss: 5.5296 - val_accuracy: 0.0609 - val_loss: 6.1153 - learning_rate: 0.0010
Epoch 4/100
2486/2487 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0867 - loss: 5.4737
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0884 - loss: 5.4602 - val_accuracy: 0.0612 - val_loss: 6.1253 - learning_rate: 0.0010
Epoch 5/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.0938 - loss: 5.3425 - val_accuracy: 0.0596 - val_loss: 6.1652 - learning_rate: 5.0000e-04
Epoch 6/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accura

In [90]:
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 2s - 2ms/step - accuracy: 0.0519 - loss: 6.0855

Test accuracy: 0.0519


In [91]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(64, activation="gelu", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="gelu", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="gelu", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(64, activation="gelu", name="deep3")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 64)        │        384 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ deep3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 128)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1625)      │    209,625 │ wide_deep_concat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 276,761 (1.06 MB)

 Trainable params: 275,993 (1.05 MB)

 Non-trainable params: 768 (3.00 KB)

In [92]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [93]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=100, callbacks = callbacks, validation_split=0.3)

Epoch 1/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0692 - loss: 6.0516 - val_accuracy: 0.0648 - val_loss: 6.0725 - learning_rate: 0.0010
Epoch 2/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.0849 - loss: 5.6439 - val_accuracy: 0.0623 - val_loss: 6.0723 - learning_rate: 0.0010
Epoch 3/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0907 - loss: 5.4709 - val_accuracy: 0.0602 - val_loss: 6.1052 - learning_rate: 0.0010
Epoch 4/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0943 - loss: 5.3612 - val_accuracy: 0.0597 - val_loss: 6.1267 - learning_rate: 0.0010
Epoch 5/100
2475/2487 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0948 - loss: 5.2912
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.0961 - loss: 5.2885 - val_accuracy: 0.0569 - val_loss: 6.1346 - learning_rate: 0.0010
Epoch 6/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0

In [94]:
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 2s - 2ms/step - accuracy: 0.0577 - loss: 6.0197

Test accuracy: 0.0577


In [95]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(64, activation="swish", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="swish", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="swish", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(64, activation="swish", name="deep3")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 64)        │        384 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ deep3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 128)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1625)      │    209,625 │ wide_deep_concat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 276,761 (1.06 MB)

 Trainable params: 275,993 (1.05 MB)

 Non-trainable params: 768 (3.00 KB)

In [96]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [97]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=100, callbacks = callbacks, validation_split=0.3)

Epoch 1/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0694 - loss: 6.0516 - val_accuracy: 0.0609 - val_loss: 6.0759 - learning_rate: 0.0010
Epoch 2/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.0849 - loss: 5.6413 - val_accuracy: 0.0575 - val_loss: 6.0761 - learning_rate: 0.0010
Epoch 3/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0908 - loss: 5.4795 - val_accuracy: 0.0570 - val_loss: 6.1047 - learning_rate: 0.0010
Epoch 4/100
2476/2487 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0919 - loss: 5.3901
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0939 - loss: 5.3786 - val_accuracy: 0.0563 - val_loss: 6.1330 - learning_rate: 0.0010
Epoch 5/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0996 - loss: 5.2577 - val_accuracy: 0.0567 - val_loss: 6.1464 - learning_rate: 5.0000e-04
Epoch 6/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accur

In [98]:
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 3s - 2ms/step - accuracy: 0.0545 - loss: 6.0614

Test accuracy: 0.0545


In [99]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

n_wide  = X_train_wide.shape[1]   # 5
n_deep  = X_train_deep.shape[1]   # 93
n_classes = 1625

input_wide = tf.keras.layers.Input(shape=(n_wide,),  name="wide_input")
input_deep = tf.keras.layers.Input(shape=(n_deep,),  name="deep_input")

# A single dense layer lets the model learn direct feature→class weights
wide_out = tf.keras.layers.Dense(64, activation="mish", name="wide_dense")(input_wide)

deep = tf.keras.layers.Dense(256, activation="mish", name="deep1")(input_deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(128, activation="mish", name="deep2")(deep)
deep = tf.keras.layers.BatchNormalization()(deep)
deep = tf.keras.layers.Dropout(0.3)(deep)

deep = tf.keras.layers.Dense(64, activation="mish", name="deep3")(deep)
deep_out = tf.keras.layers.Dropout(0.2)(deep)

# Concatenates then classifies
concat = tf.keras.layers.Concatenate(name="wide_deep_concat")([wide_out, deep_out])
output = tf.keras.layers.Dense(n_classes, activation="softmax", name="output")(concat)

model = tf.keras.Model(inputs=[input_wide, input_deep], outputs=output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ deep_input          │ (None, 93)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep1 (Dense)       │ (None, 256)       │     24,064 │ deep_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ deep1[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep2 (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ deep2[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep3 (Dense)       │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_dense (Dense)  │ (None, 64)        │        384 │ wide_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ deep3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wide_deep_concat    │ (None, 128)       │          0 │ wide_dense[0][0], │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1625)      │    209,625 │ wide_deep_concat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 276,761 (1.06 MB)

 Trainable params: 275,993 (1.05 MB)

 Non-trainable params: 768 (3.00 KB)

In [100]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [101]:
history = model.fit([X_train_wide, X_train_deep], y_train, epochs=100, callbacks = callbacks, validation_split=0.3)

Epoch 1/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0699 - loss: 6.0384 - val_accuracy: 0.0592 - val_loss: 6.1049 - learning_rate: 0.0010
Epoch 2/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0857 - loss: 5.6304 - val_accuracy: 0.0575 - val_loss: 6.0853 - learning_rate: 0.0010
Epoch 3/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0897 - loss: 5.4804 - val_accuracy: 0.0572 - val_loss: 6.1022 - learning_rate: 0.0010
Epoch 4/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0934 - loss: 5.3864 - val_accuracy: 0.0582 - val_loss: 6.1222 - learning_rate: 0.0010
Epoch 5/100
2484/2487 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0937 - loss: 5.3284
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.0960 - loss: 5.3167 - val_accuracy: 0.0603 - val_loss: 6.1540 - learning_rate: 0.0010
Epoch 6/100
2487/2487 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy:

In [102]:
test_loss, test_acc = model.evaluate([X_test_wide, X_test_deep], y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

1355/1355 - 2s - 2ms/step - accuracy: 0.0588 - loss: 6.0140

Test accuracy: 0.0588


## Final Model Metrics

Full classification report and per-class TP/TN/FP/FN for the best Wide-Deep model (most recent `model` in memory). Shows top 20 most frequent categories — reporting all 1,625 classes is unwieldy given class sparsity.

In [103]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Generate predictions from the two-input Wide-Deep model
y_pred_nn = np.argmax(model.predict([X_test_wide, X_test_deep]), axis=1)

print(f"Test Accuracy: {(y_test == y_pred_nn).mean():.4f}")
print()

# Classification report for top 20 most frequent categories
top_20 = pd.Series(y_test).value_counts().head(20).index.tolist()
mask_20 = pd.Series(y_test).isin(top_20)
print("Classification Report - Top 20 Most Frequent Categories:")
print(classification_report(
    y_test[mask_20], y_pred_nn[mask_20],
    labels=top_20, zero_division=0
))

# TP / TN / FP / FN for top 20 classes
n_cls = int(y_test.max()) + 1
cm_nn = confusion_matrix(y_test, y_pred_nn, labels=list(range(n_cls)))
total_nn = len(y_test)
rows_nn = []
for cls in top_20:
    tp = int(cm_nn[cls, cls])
    fp = int(cm_nn[:, cls].sum() - tp)
    fn = int(cm_nn[cls, :].sum() - tp)
    tn = int(total_nn - tp - fp - fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
    rows_nn.append({"Class (encoded)": cls, "TP": tp, "TN": tn, "FP": fp, "FN": fn,
                    "Precision": round(prec,4), "Recall": round(rec,4), "F1": round(f1,4)})

metrics_nn = pd.DataFrame(rows_nn).set_index("Class (encoded)")
print("Per-class TP / TN / FP / FN - Neural Network (Wide-Deep), Top 20 Categories:")
print(metrics_nn.to_string())

from sklearn.metrics import f1_score
print()
print(f"F1 Macro    (unweighted avg over classes): {f1_score(y_test, y_pred_nn, average='macro',    zero_division=0):.4f}")
print(f"F1 Weighted (weighted by class support):   {f1_score(y_test, y_pred_nn, average='weighted', zero_division=0):.4f}")
print(f"F1 Micro    (global TP / FP / FN totals):  {f1_score(y_test, y_pred_nn, average='micro',    zero_division=0):.4f}")


1355/1355 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Test Accuracy: 0.0588

Classification Report - Top 20 Most Frequent Categories:
              precision    recall  f1-score   support

           2       0.29      0.63      0.40      1715
        1044       0.32      0.30      0.31       864
        1248       0.13      0.04      0.06       823
         983       0.24      0.28      0.25       694
         906       0.25      0.12      0.16       491
        1015       0.10      0.11      0.11       423
         499       0.28      0.09      0.14       421
        1276       0.17      0.00      0.01       419
         731       0.00      0.00      0.00       371
        1251       0.34      0.18      0.24       370
        1271       0.38      0.08      0.13       358
         278       0.00      0.00      0.00       320
        1465       0.27      0.07      0.11       305
         595       0.20      0.01      0.01       289
        1560       0.33      0.02      0.03       285
        1050